# Цифровой двойник бульдозера по телеметрии Unity

Цель — восстановить состояние бульдозера на вычислительном узле, отделённом от симулятора. Unity выступает источником измерений. Каждые 20 мс она записывает положение шасси и число камней, перемещённых более чем на 0,45 м по горизонтали относительно начала опыта.

Блокнот совместим с Google Colab и Jupyter. Сохранённые ниже результаты получены локально. Для запуска в Colab достаточно открыть этот ipynb и выполнить все ячейки: контрольная запись встроена. Для анализа другого прогона положите новый yard_telemetry.csv рядом с блокнотом. Схема передачи — пакет измерений → проверка номеров и времени → восстановление состояния → расчёт показателей.

In [ ]:
from pathlib import Path
import csv
from io import StringIO
from math import hypot
from statistics import median

embedded = 'step,time_s,x_m,z_m,displaced_rocks\n0,0.00,-31.0000,-8.0000,0\n1,0.02,-31.0000,-7.9360,0\n2,0.04,-31.0000,-7.8720,0\n3,0.06,-31.0000,-7.8080,0\n4,0.08,-31.0000,-7.7440,0\n5,0.10,-31.0000,-7.6800,0\n6,0.12,-31.0000,-7.6160,0\n7,0.14,-31.0000,-7.5520,0\n8,0.16,-31.0000,-7.4880,0\n9,0.18,-31.0000,-7.4240,0\n10,0.20,-31.0000,-7.3600,0\n11,0.22,-31.0000,-7.2960,0\n12,0.24,-31.0000,-7.2320,0\n13,0.26,-31.0000,-7.1680,0\n14,0.28,-31.0000,-7.1040,0\n15,0.30,-31.0000,-7.0400,0\n16,0.32,-31.0000,-6.9760,0\n17,0.34,-31.0000,-6.9120,0\n18,0.36,-31.0000,-6.8480,0\n19,0.38,-31.0000,-6.7840,0\n20,0.40,-31.0000,-6.7200,0\n21,0.42,-31.0000,-6.6560,0\n22,0.44,-31.0000,-6.5920,0\n23,0.46,-31.0000,-6.5280,0\n24,0.48,-31.0000,-6.4640,0\n25,0.50,-31.0000,-6.4000,0\n26,0.52,-31.0000,-6.3360,0\n27,0.54,-31.0000,-6.2720,0\n28,0.56,-31.0000,-6.2080,0\n29,0.58,-31.0000,-6.1440,0\n30,0.60,-31.0000,-6.0800,0\n31,0.62,-31.0000,-6.0160,0\n32,0.64,-31.0000,-5.9520,0\n33,0.66,-31.0000,-5.8880,0\n34,0.68,-31.0000,-5.8240,5\n35,0.70,-31.0000,-5.7600,10\n36,0.72,-31.0000,-5.6960,15\n37,0.74,-31.0000,-5.6320,20\n38,0.76,-31.0000,-5.5680,25\n39,0.78,-31.0000,-5.5040,28\n40,0.80,-31.0000,-5.4400,28\n41,0.82,-31.0000,-5.3760,28\n42,0.84,-31.0000,-5.3120,29\n43,0.86,-31.0000,-5.2480,29\n44,0.88,-31.0000,-5.1840,30\n45,0.90,-31.0000,-5.1200,31\n46,0.92,-31.0000,-5.0560,31\n47,0.94,-31.0000,-4.9920,31\n48,0.96,-31.0000,-4.9280,31\n49,0.98,-31.0000,-4.8640,31\n50,1.00,-31.0000,-4.8000,31\n51,1.02,-31.0000,-4.7360,31\n52,1.04,-31.0000,-4.6720,31\n53,1.06,-31.0000,-4.6080,31\n54,1.08,-31.0000,-4.5440,31\n55,1.10,-31.0000,-4.4800,31\n56,1.12,-31.0000,-4.4160,31\n57,1.14,-31.0000,-4.3520,31\n58,1.16,-31.0000,-4.2880,31\n59,1.18,-31.0000,-4.2240,31\n60,1.20,-31.0000,-4.1600,31\n61,1.22,-31.0000,-4.0960,31\n62,1.24,-31.0000,-4.0320,31\n63,1.26,-31.0000,-3.9680,31\n64,1.28,-31.0000,-3.9040,31\n65,1.30,-31.0000,-3.8400,31\n66,1.32,-31.0000,-3.7760,31\n67,1.34,-31.0000,-3.7120,31\n68,1.36,-31.0000,-3.6480,31\n69,1.38,-31.0000,-3.5840,31\n70,1.40,-31.0000,-3.5200,31\n71,1.42,-31.0000,-3.4560,31\n72,1.44,-31.0000,-3.3920,31\n73,1.46,-31.0000,-3.3280,31\n74,1.48,-31.0000,-3.2640,31\n75,1.50,-31.0000,-3.2000,31\n76,1.52,-31.0000,-3.1360,31\n77,1.54,-31.0000,-3.0720,32\n78,1.56,-31.0000,-3.0080,32\n79,1.58,-31.0000,-2.9440,32\n80,1.60,-31.0000,-2.8800,32\n81,1.62,-31.0000,-2.8160,32\n82,1.64,-31.0000,-2.7520,32\n83,1.66,-31.0000,-2.6880,32\n84,1.68,-31.0000,-2.6240,32\n85,1.70,-31.0000,-2.5600,32\n86,1.72,-31.0000,-2.4960,32\n87,1.74,-31.0000,-2.4320,32\n88,1.76,-31.0000,-2.3680,32\n89,1.78,-31.0000,-2.3040,32\n90,1.80,-31.0000,-2.2400,32\n'
file = Path('yard_telemetry.csv')
source = file.read_text(encoding='utf-8-sig') if file.is_file() else embedded
rows = list(csv.DictReader(StringIO(source)))
print(f'Принято отсчётов: {len(rows)}')

In [ ]:
def replay(records):
    samples = {}
    for row in records:
        key = int(row['step'])
        observation = (float(row['time_s']), float(row['x_m']), float(row['z_m']), int(row['displaced_rocks']))
        if key in samples and samples[key] != observation:
            raise ValueError(f'Противоречивый повтор отсчёта {key}')
        samples[key] = observation
    indices = sorted(samples)
    if not indices or indices != list(range(indices[-1] + 1)):
        raise ValueError('Пропущен шаг телеметрии')
    ordered = [samples[i] for i in indices]
    if any(b[0] <= a[0] for a, b in zip(ordered, ordered[1:])):
        raise ValueError('Время должно возрастать')
    return ordered

history = replay(rows)
assert history == replay(list(reversed(rows)) + rows[:3])
try:
    replay(rows[:4] + rows[5:])
except ValueError as error:
    print('Проверка потери пакета:', error)
else:
    raise AssertionError('Потеря осталась незамеченной')
print('Повторы и изменение порядка доставки не меняют состояние двойника.')

In [ ]:
speeds = []
distance = 0.0
for a, b in zip(history, history[1:]):
    step_length = hypot(b[1] - a[1], b[2] - a[2])
    distance += step_length
    speeds.append(step_length / (b[0] - a[0]))
first_contact = next((t for t, x, z, moved in history if moved > 0), None)
time, x, z, rocks = history[-1]
print(f'Время {time:.2f} с; пройдено {distance:.3f} м; медианная скорость {median(speeds):.3f} м/с')
print(f'Последнее состояние: X={x:.3f} м, Z={z:.3f} м. Сдвинуто камней: {rocks}.')
print(f'Первое превышение порога смещения: {first_contact} с.')
print('Время, c    Z, м     камней')
for t, x, z, moved in history[::10]:
    print(f'{t:7.2f} {z:8.3f} {moved:8d}')

### Интерпретация
Рост числа сдвинутых камней отражает работу отвала. Начало движения машины и первый камень, преодолевший порог, могут приходиться на разные шаги: отвалу нужно доехать до насыпи. Пороговое число описывает состояние в каждый момент; один и тот же камень повторно не суммируется. Блокнот восстанавливает состояние из сохранённой телеметрии. Задержки реальной сети и выход на физическую стройплощадку экспериментом не измерялись.